# Staple Paths: Weighting, Reversed Pieces, Cut at M and Piece Length

Fits the committor (`CommittorNE`, CuPy, NPNEq) to one staple simulation under several data preparations and compares the results, against the exact committor when one is available. Every configuration uses the same paths, the same fit settings and the same random seed; only the preparation of the data differs. The simulation is chosen with a preset (`SYSTEM`): the 1D flat Brownian run or the 2D Brownian Z run, or one you add.

**What is varied** (one entry of `CONFIGS` per fit):

| Key | Values | Meaning |
|---|---|---|
| `weights` | `rejections`, `staple`, `none` | MC path weights from tistools (`tis_data.read_path_info`): `rejections` counts a path once per cycle it stays the current path; `staple` also applies the high-acceptance factor for shot LML/RMR paths; `none` counts every accepted path once |
| `reversed` | `False`, `True` | also use every path's mirrored piece (from its start to its last M crossing, read backwards); needs time-reversible dynamics at equilibrium |
| `cut` | `True`, `False` | cut every path at its first frame past M. Without the cut, the frames before that crossing are conditioned on crossing M later (biased; for comparison only, and not combinable with `reversed`) |
| `max_length` | `None` or N frames | limit every piece (after the cut) to N frames |
| `long_pieces` | `truncate`, `discard` | what to do with longer pieces: keep their first N frames, or drop them |

**Piece length.** Truncating a piece a fixed number of frames after its start is a stopping rule (stop at whichever comes first: the path's own end or frame N). A truncated piece is therefore still a valid stopped segment and should not bias the committor; it only removes data, and with it the longest lags of the stopped Z_q. Discarding long pieces instead selects pieces on how they continue: the long ones are those that linger between the interfaces. That biases the data, and it is in the list to show by how much. The limit applies to each piece separately, so with `reversed` the forward and mirrored piece of a path are truncated or dropped independently.

**Yardstick.** Each fit's stopped Z_q (`zqs_z`, the worst |Z_q / standard error|) is computed on its own data, so a biased data set can still pass its own test. The common yardstick is the exact committor (`q_exact_of`): fit − exact along λ (and per (λ, CV 1) bin for 2D data), and the RMS over frames with 0.05 < q_exact < 0.95, weighted by the path weights.

Each configuration's results are cached in `<OUT>/<name>/summary.npz`, together with its `plots_tis` and metrics figures. A configuration that already has a summary is not fitted again unless `RERUN = True`, so the comparison in section 5 can be redrawn, and configurations added, without repeating earlier fits. `OUT` contains the simulation's name, so the two systems do not share a cache.

## 1. Imports

In [ ]:
import argparse
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cupy as cp
from scipy.interpolate import RegularGridInterpolator

plt.rc('font', size=10)
plt.rc('axes', titlesize=12)
plt.rc('axes', labelsize=12)
plt.rc('legend', fontsize=9)

# Custom optimal-RC / committor package (CuPy) and the path-data loader of the examples
REPO = '/home/staff7/fbsskr/Notebooks/Students/elias/optimalrcs'
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs
import tis_data
FIG = f'{REPO}/examples/figures'

## 2. Settings

**System.** `SYSTEM` picks a preset; each preset sets:

| Name | Meaning |
|---|---|
| `TIS_DIR`, `LAMBDA_A`, `LAMBDA_B` | the run and its states A and B (`None`: first and last interface) |
| `PATHS_PER_ENSEMBLE`, `PATH_SEED` | random subset of accepted paths per ensemble (`None` = all) |
| `WARMUP_ITER`, `MAX_ITER` | iterations of the warm-up (wide envelope) and of the main run |
| `MAX_LENGTH` | piece-length limit of the truncate/discard configurations, in frames |
| `q_exact_of(X_all)` | exact committor per frame from the CVs, or `None` (the fits are then compared to `BASELINE`) |
| `EDGES`, `LABELS` | bins and axis labels: one entry for a profile along λ only (1D), two for (λ, CV 1) maps as well (`EDGES = None`: chosen from the first loaded data) |
| `CONTOURS` | `(g1, g2, V, levels)`: grey potential contours on the maps, or `None` |

Presets: `1d_flat` (`sim_istar0br10intf`, flat 1D overdamped; exact committor linear) and `2d_z` (`sim_istarz_br_g10`, 2D Brownian Z potential; exact committor on a grid). For another simulation, add a branch with the same names.

**Paths.** The subset draw only depends on the seed and on which paths have a nonzero weight, so every configuration gets the same paths (as long as its weighting gives every accepted path a nonzero weight, which is true for `rejections`, `staple` and `none`).

**Fit.** The same for every configuration: random CV per iteration, seed `FIT_SEED`. Rough cost: about 0.4 s per iteration on 20M frames, so a `2d_z` configuration takes about 2 hours, a `1d_flat` one (7.6M frames, 3000 iterations) well under one.

In [ ]:
SYSTEM = '1d_flat'                 # '1d_flat' or '2d_z'; add a preset below for another simulation
SIMS = '/home/staff7/fbsskr/Notebooks/Students/elias/data/simulations'

if SYSTEM == '1d_flat':
    # 1D flat potential, overdamped: gamma = 1, kT = 1, dt = 2e-4. Median piece about 60 frames.
    TIS_DIR = f'{SIMS}/sim_istar0br10intf'
    LAMBDA_A, LAMBDA_B = -0.1, 0.8
    PATHS_PER_ENSEMBLE = 12000     # 7.6M frames; all 33M do not fit the GPU for the stopped Z_q
    PATH_SEED = 0
    WARMUP_ITER, MAX_ITER = 0, 3000
    MAX_LENGTH = 200               # frames, for the piece-length configurations
    EDGES = (np.linspace(LAMBDA_A, LAMBDA_B, 91),)
    LABELS = ('order parameter $\\lambda$',)
    CONTOURS = None

    # Exact committor: linear, with the boundaries shifted out by the mean overshoot of a
    # discrete Brownian step (0.5826 step sizes; absorption is checked every step).
    _shift = 0.5826 * np.sqrt(2 * 1.0 * 2e-4 / 1.0)   # sqrt(2 kT dt / gamma)
    def q_exact_of(X):
        return np.clip((X[:, 0] - LAMBDA_A + _shift) / (LAMBDA_B - LAMBDA_A + 2 * _shift), 0, 1)

elif SYSTEM == '2d_z':
    # 2D Brownian Z potential sjoelbak_angle(Lx=0.3, angle=45): gamma = 10, kT = 0.05, dt = 0.002.
    TIS_DIR = f'{SIMS}/sim_istarz_br_g10'
    LAMBDA_A, LAMBDA_B = 0.05, 0.85
    PATHS_PER_ENSEMBLE = 2000
    PATH_SEED = 112
    WARMUP_ITER, MAX_ITER = 1000, 15000
    MAX_LENGTH = 1000
    EDGES = (np.linspace(0.05, 0.85, 81), np.linspace(-0.01, 0.31, 33))
    LABELS = ('order parameter $y$', '$x$')

    # Exact committor: finite-volume solution on a (y, x) grid (figures/z_brownian/committor_reference.py)
    _ref = np.load(f'{FIG}/z_brownian/committor_reference_h0.001.npz')
    _f_ref = RegularGridInterpolator((_ref['y'], _ref['x']), _ref['q'], bounds_error=False, fill_value=None)
    def q_exact_of(X):
        return np.clip(_f_ref(X[:, :2]), 0, 1)
    _in = (_ref['y'] >= -0.01) & (_ref['y'] <= 0.91)
    _V = _ref['V_eff'][_in]
    CONTOURS = (_ref['y'][_in], _ref['x'], _V, np.linspace(np.percentile(_V, 1), np.percentile(_V, 92), 14))

else:
    raise ValueError(f'unknown SYSTEM {SYSTEM!r}')

# --- shared by both systems ---
N_CVS = None                       # train on the first N CVs (None = all in order.txt)
FIT_SEED = 0                       # CV and envelope-centre draws
WARMUP_ENVELOPE, ENVELOPE = 1.0, 0.01
GAMMA = 0.1
FIT_KW = dict(ny=6, print_step=500, min_delta_x=1e-6, lazy_basis=True, save_min_metric='max_z_zq_stopped',
              metrics_print=('iter', 'delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x', 'time_elapsed'))
MIN_COUNT = 20                     # frames a bin needs to be shown
OUT = f'{FIG}/staple_compare/{os.path.basename(TIS_DIR.rstrip("/"))}_p{PATHS_PER_ENSEMBLE}_s{PATH_SEED}'
RERUN = False                      # fit again even if a configuration has a summary
SAVE_R = False                     # also save every fit's q per frame (4 bytes per frame)

## 3. Configurations

Each entry overrides the defaults `weights='rejections'`, `reversed=False`, `cut=True`, `max_length=None`, `long_pieces='truncate'`. `RUN` selects which configurations to fit or load. `BASELINE` is drawn thicker in the plots, and is the reference when there is no exact committor.

`MAX_LENGTH` comes from the preset and is in frames of the saved paths. The piece-length histogram in section 5 (from any configuration without a limit) shows how many pieces a value affects; a useful limit cuts the long tail but keeps most pieces whole.

In [ ]:
CONFIGS = [
    dict(name='fwd_rej',    weights='rejections'),
    dict(name='rev_rej',    weights='rejections', reversed=True),
    dict(name='fwd_staple', weights='staple'),
    dict(name='rev_staple', weights='staple',     reversed=True),
    dict(name='fwd_none',   weights='none'),
    dict(name='rev_none',   weights='none',       reversed=True),
    dict(name='nocut_rej',  weights='rejections', cut=False),
    dict(name=f'rev_rej_trunc{MAX_LENGTH}', weights='rejections', reversed=True,
         max_length=MAX_LENGTH, long_pieces='truncate'),
    dict(name=f'rev_rej_disc{MAX_LENGTH}',  weights='rejections', reversed=True,
         max_length=MAX_LENGTH, long_pieces='discard'),
]
RUN = [c['name'] for c in CONFIGS]  # or a subset, e.g. ['fwd_rej', 'rev_rej']
BASELINE = 'rev_rej'

DEFAULTS = dict(weights='rejections', reversed=False, cut=True, max_length=None, long_pieces='truncate')
CONFIGS = [{**DEFAULTS, **c} for c in CONFIGS]
for c in CONFIGS:
    if c['reversed'] and not c['cut']:
        raise ValueError(f"{c['name']}: reversed pieces need the cut at M")
assert len({c['name'] for c in CONFIGS}) == len(CONFIGS), 'configuration names must be unique'
pd.DataFrame(CONFIGS).set_index('name')

## 4. Fit Every Configuration

For every configuration in `RUN` without a cached summary:
1. the paths are read with `tis_data.prepare_data` (the same code as `load_tis_data_cupy.py`, with `--weights`, `--reversed`, `--no-cut`, `--max-length` and `--long-pieces`);
2. `CommittorNE` is fitted, with the A/B ends of the pieces as traps for the metrics (`set_fixed_traj_length_trap`; truncated pieces end between the interfaces and are not extended);
3. the summary is saved: data size, piece lengths, metrics history, fit and exact q per bin, and the RMS of fit − exact. The figures of `plots_tis` and `plots_metrics` go to `<OUT>/<name>/`.

The summary describes the final q of the run (not the one with the lowest `zqs_z`, which the fit keeps as `q.r_traj_min_sd_zq`).

In [ ]:
if q_exact_of is None:
    print('No exact committor: the fits are compared with', BASELINE)


def load_config(cfg):
    """The frames of one configuration, read with tis_data.prepare_data."""
    parser = argparse.ArgumentParser()
    tis_data.add_data_arguments(parser)
    argv = [TIS_DIR, '--weights', cfg['weights'], '--long-pieces', cfg['long_pieces']]
    if LAMBDA_A is not None:
        argv += ['--lambda-a', str(LAMBDA_A)]
    if LAMBDA_B is not None:
        argv += ['--lambda-b', str(LAMBDA_B)]
    if PATHS_PER_ENSEMBLE is not None:
        argv += ['--paths-per-ensemble', str(PATHS_PER_ENSEMBLE), '--seed', str(PATH_SEED)]
    if N_CVS is not None:
        argv += ['--n-cvs', str(N_CVS)]
    if cfg['reversed']:
        argv += ['--reversed']
    if not cfg['cut']:
        argv += ['--no-cut']
    if cfg['max_length'] is not None:
        argv += ['--max-length', str(cfg['max_length'])]
    return tis_data.prepare_data(parser.parse_args(argv))


def fit(data):
    """Warm-up and main run of CommittorNE on one configuration's frames."""
    np.random.seed(FIT_SEED); cp.random.seed(FIT_SEED)
    q = optimalrcs.CommittorNE(
        boundary0=data.boundary0, boundary1=data.boundary1,
        i_traj=data.i_traj, t_traj=data.t_traj,
        path_weights=data.path_weights,
        group_traj=data.group_traj,    # forward and mirrored piece of a path: one group
    )
    q.set_fixed_traj_length_trap(data.boundary0 | data.boundary1, 2 ** 16)   # A/B ends as traps, for the metrics
    X_gpu = cp.asarray(data.X, dtype=q.prec)

    def comp_y():
        return X_gpu[:, np.random.randint(X_gpu.shape[1])]

    if WARMUP_ITER > 0:
        optimalrcs.envelope_scale = WARMUP_ENVELOPE
        q.fit_transform(comp_y, gamma=GAMMA, max_iter=WARMUP_ITER, **FIT_KW)
    optimalrcs.envelope_scale = ENVELOPE
    q.fit_transform(comp_y, gamma=GAMMA, max_iter=MAX_ITER, **FIT_KW)
    return q


def binned(coords, edges, val, w, keep):
    """Weighted mean of `val` per bin of `coords` over the frames in `keep`;
    NaN in bins with fewer than MIN_COUNT frames."""
    sample = np.column_stack([c[keep] for c in coords])
    n = np.histogramdd(sample, bins=edges)[0]
    sw = np.histogramdd(sample, bins=edges, weights=w[keep])[0]
    s = np.histogramdd(sample, bins=edges, weights=(w * val)[keep])[0]
    return np.where(n >= MIN_COUNT, s / np.where(sw > 0, sw, 1), np.nan)


def get_edges(data):
    """EDGES, or bins over the free frames of the first data set (then kept for all):
    lambda only for one CV, (lambda, CV 1) for more."""
    global EDGES
    if EDGES is None:
        free = ~(data.boundary0 | data.boundary1)
        X = data.X_all[free]
        EDGES = tuple(np.linspace(*np.percentile(X[:, k], [0.5, 99.5]), 81 if k == 0 else 33)
                      for k in range(min(2, X.shape[1])))
    return EDGES


def summarize(q, data, cfg, t_fit):
    """Everything section 5 needs from one fit, as arrays for np.savez."""
    r = cp.asnumpy(q.r_traj)
    X, w = data.X_all, data.path_weights
    free = ~(data.boundary0 | data.boundary1)
    edges = get_edges(data)
    s = dict(cfg=json.dumps(cfg), t_fit=t_fit, n_frames=len(r),
             n_pieces=len(np.unique(data.i_traj)), n_paths=len(np.unique(data.group_traj)),
             piece_lengths=np.bincount(data.i_traj).astype(np.int32),
             edges_lam=edges[0])
    for key in ('iter', 'delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x'):
        s[f'hist_{key}'] = np.array([float(v) for v in q.metrics_history.get(key, [])])
    s['prof_fit'] = binned((X[:, 0],), edges[:1], r, w, free)
    if len(edges) > 1 and X.shape[1] > 1:
        s['edges_cv1'] = edges[1]
        s['map_fit'] = binned((X[:, 0], X[:, 1]), edges, r, w, free)
    if q_exact_of is not None:
        qe = q_exact_of(X)
        trans = free & (qe > 0.05) & (qe < 0.95)
        s['rms_frames'] = np.sqrt(np.sum(w[trans] * (r[trans] - qe[trans]) ** 2) / np.sum(w[trans]))
        s['prof_exact'] = binned((X[:, 0],), edges[:1], qe, w, free)
        if 'map_fit' in s:
            s['map_exact'] = binned((X[:, 0], X[:, 1]), edges, qe, w, free)
        # RMS per bin over the finest binning: (lambda, CV 1) maps if there are any, else the profile
        fit_b, ex_b = (s['map_fit'], s['map_exact']) if 'map_fit' in s else (s['prof_fit'], s['prof_exact'])
        bins = (ex_b > 0.05) & (ex_b < 0.95)
        s['rms_bins'] = np.sqrt(np.nanmean((fit_b - ex_b)[bins] ** 2))
    return s


def save_fit_plots(q, folder):
    """plots_tis and plots_metrics of the fit, saved and closed (not shown)."""
    show, plt.show = plt.show, (lambda *a, **k: None)
    try:
        q.plots_tis()
        plt.gcf().savefig(f'{folder}/tis.png', dpi=100)
        q.plots_metrics(['delta_r2', 'max_z_zq_stopped', 'delta_x'])
        plt.gcf().savefig(f'{folder}/metrics.png', dpi=100)
    finally:
        plt.show = show
        plt.close('all')


def load_summary(path):
    with np.load(path) as d:
        return {k: (d[k][()] if d[k].ndim == 0 else d[k]) for k in d.files}

In [ ]:
summaries = {}
for cfg in CONFIGS:
    name = cfg['name']
    if name not in RUN:
        continue
    folder = f'{OUT}/{name}'
    path = f'{folder}/summary.npz'
    if os.path.exists(path) and not RERUN:
        summaries[name] = load_summary(path)
        if EDGES is None:   # keep the bins of the cached fits
            s = summaries[name]
            EDGES = (s['edges_lam'],) + ((s['edges_cv1'],) if 'edges_cv1' in s else ())
        print(f'{name}: loaded from {path}')
        continue
    print(f'\n=== {name}: {cfg} ===', flush=True)
    os.makedirs(folder, exist_ok=True)
    data = load_config(cfg)
    t0 = time.time()
    q = fit(data)
    s = summarize(q, data, cfg, time.time() - t0)
    np.savez(path, **s)
    save_fit_plots(q, folder)
    if SAVE_R:
        np.save(f'{folder}/q_traj.npy', cp.asnumpy(q.r_traj).astype(np.float32))
    summaries[name] = s
    print(f'{name}: done in {s["t_fit"] / 3600:.2f} h, zqs_z {s["hist_max_z_zq_stopped"][-1]:.2f}'
          + (f', RMS(fit - exact) {s["rms_frames"]:.4f}' if 'rms_frames' in s else ''), flush=True)
    del q, data
    cp.get_default_memory_pool().free_all_blocks()

## 5. Comparison

**Table.** Data size, the final and lowest `zqs_z` of the run (own data; about 2 or less is consistent), and with the exact committor the RMS of fit − exact over frames with 0.05 < q_exact < 0.95 (path-weighted), and over the bins in that range ((λ, CV 1) bins for 2D data, λ bins for 1D).

In [ ]:
rows = []
for name, s in summaries.items():
    cfg = json.loads(str(s['cfg']))
    z = s['hist_max_z_zq_stopped']
    rows.append(dict(name=name, weights=cfg['weights'], reversed=cfg['reversed'], cut=cfg['cut'],
                     max_length=cfg['max_length'], long_pieces=cfg['long_pieces'] if cfg['max_length'] else '',
                     paths=int(s['n_paths']), pieces=int(s['n_pieces']), frames=int(s['n_frames']),
                     zqs_z_final=z[-1] if len(z) else np.nan, zqs_z_min=z[1:].min() if len(z) > 1 else np.nan,
                     rms_frames=s.get('rms_frames', np.nan), rms_bins=s.get('rms_bins', np.nan),
                     fit_hours=s['t_fit'] / 3600))
table = pd.DataFrame(rows).set_index('name')
os.makedirs(OUT, exist_ok=True)
table.to_csv(f'{OUT}/comparison.csv')
table.style.format({'zqs_z_final': '{:.2f}', 'zqs_z_min': '{:.2f}', 'rms_frames': '{:.4f}',
                    'rms_bins': '{:.4f}', 'fit_hours': '{:.2f}', 'frames': '{:,}'})

**Piece lengths and convergence.** Left: the distribution of piece lengths of every configuration, with `MAX_LENGTH`; a limit removes the tail to its right (truncated pieces pile up at the limit). Right: `zqs_z` over the iterations (warm-up included).

In [ ]:
names = list(summaries)
colors = {n: plt.cm.tab10(i % 10) for i, n in enumerate(names)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
longest = max(s['piece_lengths'].max() for s in summaries.values())
bins = np.logspace(0, np.log10(longest + 1), 60)
for n, s in summaries.items():
    ax1.hist(s['piece_lengths'], bins=bins, histtype='step', color=colors[n], label=n,
             lw=2 if n == BASELINE else 1)
ax1.axvline(MAX_LENGTH, color='k', ls='--', lw=1, label=f'MAX_LENGTH = {MAX_LENGTH}')
ax1.set(xscale='log', yscale='log', xlabel='piece length (frames)', ylabel='pieces')
ax1.legend(ncol=2)

for n, s in summaries.items():
    it, z = s['hist_iter'], s['hist_max_z_zq_stopped']
    if len(z) > 1:
        ax2.plot(it[1:], z[1:], color=colors[n], label=n, lw=2 if n == BASELINE else 1)
ax2.axhline(2, color='k', ls=':', lw=1)
ax2.set(yscale='log', xlabel='iteration', ylabel='zqs_z (own data)')
ax2.legend(ncol=2)
fig.tight_layout()
fig.savefig(f'{OUT}/lengths_convergence.png', dpi=100)
plt.show()

**Committor along λ** (both systems). Top: the path-weighted mean q per λ bin (free frames) of every fit; dashed black: the exact committor averaged the same way over the `BASELINE` frames. Bottom: fit − exact, each on its own frames (so differences in which frames a configuration has do not show up as an error), or fit − `BASELINE` without the exact committor. For 1D data this is the full comparison; for 2D data the mean over CV 1 can hide errors of opposite sign, which the maps below show.

In [ ]:
has_ref = all('prof_exact' in s for s in summaries.values())
base = summaries.get(BASELINE, next(iter(summaries.values())))
e = base['edges_lam']
mid = (e[1:] + e[:-1]) / 2

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for n, s in summaries.items():
    lw = 2 if n == BASELINE else 1
    ax1.plot(mid, s['prof_fit'], color=colors[n], lw=lw, label=n)
    diff = s['prof_fit'] - (s['prof_exact'] if has_ref else base['prof_fit'])
    ax2.plot(mid, diff, color=colors[n], lw=lw, label=n)
if has_ref:
    ax1.plot(mid, base['prof_exact'], 'k--', lw=1.5, label='exact')
ax1.set(ylabel='$q$ (mean per bin)')
ax1.legend(ncol=2)
ax1.grid(alpha=0.3)
ax2.axhline(0, color='k', lw=0.8)
ax2.set(xlabel=LABELS[0], ylabel='fit $-$ exact' if has_ref else f'fit $-$ {BASELINE}')
ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f'{OUT}/profiles.png', dpi=100)
plt.show()

**Maps** (2D data only: two entries in `EDGES`). One panel per configuration: fit − exact per (λ, CV 1) bin with its RMS over the bins where 0.05 < q_exact < 0.95 (or fit − `BASELINE` without the exact committor), plus the exact committor (or the `BASELINE` fit) in the first panel. Grey: `CONTOURS` of the potential; dashed: q = 0.1, 0.5, 0.9 of the exact committor.

In [ ]:
DIFF_LIM = 0.15                    # colour range of the difference maps

if all('map_fit' in s for s in summaries.values()):
    e1, e2 = base['edges_lam'], base['edges_cv1']
    m1, m2 = (e1[1:] + e1[:-1]) / 2, (e2[1:] + e2[:-1]) / 2
    ref_map = base['map_exact'] if has_ref else base['map_fit']
    panels = [(ref_map, 'exact committor' if has_ref else f'{BASELINE} (fit)', 'coolwarm', (0, 1))]
    for n, s in summaries.items():
        if has_ref:
            panels.append((s['map_fit'] - s['map_exact'], f'{n} $-$ exact, RMS {s["rms_bins"]:.4f}',
                           'PuOr', (-DIFF_LIM, DIFF_LIM)))
        elif n != BASELINE:
            panels.append((s['map_fit'] - base['map_fit'], f'{n} $-$ {BASELINE}', 'PuOr', (-DIFF_LIM, DIFF_LIM)))

    n_cols = 3
    n_rows = -(-len(panels) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 3.8 * n_rows), squeeze=False)
    for ax, (M, title, cmap, (lo, hi)) in zip(axes.ravel(), panels):
        im = ax.pcolormesh(e1, e2, M.T, cmap=cmap, vmin=lo, vmax=hi)
        if CONTOURS is not None:
            g1, g2, V, lev = CONTOURS
            ax.contour(g1, g2, V.T, levels=lev, colors='0.45', linewidths=0.5)
        ax.contour(m1, m2, np.nan_to_num(ref_map, nan=0.5).T, levels=[0.1, 0.5, 0.9],
                   colors='k', linewidths=0.8, linestyles='--')
        ax.set(title=title, xlabel=LABELS[0], ylabel=LABELS[1] if len(LABELS) > 1 else 'CV 1',
               xlim=(e1[0], e1[-1]), ylim=(e2[0], e2[-1]))
        fig.colorbar(im, ax=ax)
    for ax in axes.ravel()[len(panels):]:
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(f'{OUT}/maps.png', dpi=100)
    plt.show()
else:
    print('Profile along lambda only (one bin axis in EDGES): no maps.')

**Per-configuration validation.** The stopped Z_q, short-lag Z_q and stopped observed-vs-predicted plots of each fit (`plots_tis`, on its own data) are in `<OUT>/<name>/tis.png`, its convergence history in `<OUT>/<name>/metrics.png`.

**What to look for.**
- *Weighting:* with the cut and the same pieces, only the path weights differ. The weighting that matches the staple sampling should give the smallest fit − exact; a wrong one shifts q where ensembles with different weights overlap.
- *Reversed:* the mirrored pieces recover the frames the cut drops, near the start of each ensemble's window. If the forward-only fits are off mainly close to the interfaces where the pieces start, that is the missing data.
- *No cut:* the frames before the first M crossing are conditioned on crossing M, which pushes q up in front of M. Expect a systematic error even when its own `zqs_z` looks fine (on `1d_flat`, earlier runs gave RMS 0.12–0.13 without the cut against 0.005–0.024 with it).
- *Truncate vs discard:* truncation should agree with its untruncated configuration within the seed-to-seed spread (run two `FIT_SEED`s to see that spread); discarding drops the slow pieces and should show a bias.